# 07 — Baseline NLP Model: TF-IDF + LinearSVC

## Goal
Train a strong lightweight baseline before fine-tuning a transformer.

**What happens using this code?** Character n-grams help capture Sinhala text without depending on word segmentation; word n-grams can help with English words and phrases. LinearSVC combines these signals to classify a question.

In [ ]:
from pathlib import Path
import json, pandas as pd, joblib
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "datasets/processed"
train = pd.read_csv(DATA / "paddy_train.csv", keep_default_na=False, encoding="utf-8-sig")
val = pd.read_csv(DATA / "paddy_validation.csv", keep_default_na=False, encoding="utf-8-sig")

features = FeatureUnion([
    ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5), min_df=2, max_features=250000, sublinear_tf=True)),
    ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=100000, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")),
])
model = Pipeline([("features", features), ("classifier", LinearSVC(class_weight="balanced", random_state=42))])
model.fit(train["question"], train["label"])
pred = model.predict(val["question"])
precision, recall, f1_macro, _ = precision_recall_fscore_support(val["label"], pred, average="macro", zero_division=0)
metrics = {"model":"tfidf_linear_svc", "accuracy":float(accuracy_score(val["label"], pred)), "precision_macro":float(precision), "recall_macro":float(recall), "f1_macro":float(f1_macro)}
print(json.dumps(metrics, indent=2))
print(classification_report(val["label"], pred, zero_division=0))

model_dir = ROOT / "models/problem_classifier"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_dir / "tfidf_linear_svc.joblib")
(ROOT / "models/model_evaluation_results/baseline_validation_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
pd.DataFrame({"question":val["question"], "language":val["language"], "issue_id":val["issue_id"], "true_label":val["label"], "predicted_label":pred}).to_csv(ROOT / "models/model_evaluation_results/baseline_validation_predictions.csv", index=False, encoding="utf-8-sig")
print("Saved baseline model and validation results.")

**Why start here?** A simple baseline is fast and provides a fair comparison. The multilingual transformer must beat this baseline on held-out validation data to justify its extra compute cost.